# Actividad 2.15 — Comparación con el conocimiento publicado por Zhu

**Análisis comparativo de enfoques físico-experimentales y predictivos basados en datos para el estudio del comportamiento de bombas electrosumergibles bajo condiciones de flujo gas-líquido.**

Una bomba electrosumergible (ESP, *electrical submersible pump*) eleva la presión de la corriente de un pozo. Cuando entra gas libre, ese incremento baja y la operación puede inestabilizarse. Este cuaderno no compite con Jianjun Zhu. Lee lo que sus textos locales permiten sostener y lo contrasta con los resultados ya publicados de la actividad 2.14.

No reentrena M1–M5. No modifica el Excel ni los CSV. No implementa un modelo híbrido de física e inteligencia artificial. El informe autónomo es `docs/estudio_comparativo_zhu_2_15.md`. La matriz es `docs/matriz_evidencias_2_15.csv`.

La actividad 2.14 está cerrada desde el 16 de octubre de 2026. Esta actividad queda abierta.


## F.1 Introducción

El problema físico es el desempeño hidráulico con flujo gas-líquido. Zhu y sus colaboradores midieron curvas de incremento de presión, distinguieron ensayos de surging y de mapping, y propusieron mapas de patrón de flujo. Este laboratorio, sobre la serie publicada sin isopropanol, preguntó cuánto del incremento de la etapa 3 se puede estimar con predictores estadísticos y un protocolo de curvas.

Comprender un régimen no es lo mismo que predecir un número en libras por pulgada cuadrada (psi). Las dos preguntas se complementan. Ninguna sustituye a la otra. Las siglas que siguen se definen en su primera aparición: GVF es la fracción volumétrica de gas; Qgd es el caudal de gas normalizado; MAE es el error absoluto medio.


## F.2 Enfoque de Zhu, por documento

**Artículo de 2019** (DOI 10.1016/j.petrol.2019.05.059), leído en `docs/esp_gassy_flow_paper.pdf`. El resumen relaciona la degradación y el surging con la fracción de gas en la admisión y con la velocidad. El modelo traza fronteras entre burbuja dispersa, burbuja, flujo intermitente y flujo segregado. Surging deja fijo el caudal de líquido y aumenta el de gas. Mapping deja fijo el gas y reduce el líquido. La comparación publicada usa quiebres de la curva de altura frente a caudal, no un MAE del incremento punto a punto. En la página 3 del PDF, Qgd se describe como un caudal de gas adimensional.

**Tesis de 2017**, leída en las páginas 20 a 80 del PDF local, no en sus 250 páginas. La bomba es una TE-2700 de 14 etapas, serie 538. El punto de mejor eficiencia (BEP, *best efficiency point*) es 2700 barriles por día (bpd) a 3500 revoluciones por minuto (rpm), con 22,5 psi de incremento de etapa. El caudal abierto a 3500 rpm es 4900 bpd. Los sensores de presión citados son Rosemount 2051S; la diferencia de etapas, Rosemount 3051S. La ecuación (2.5) es GVF = Qg / (Qg + Ql). La ecuación (2.7) es Qgd = Qg / Qmax, con Qmax el caudal abierto de líquido.

**Revisión de 2018** en *Energies* (DOI 10.3390/en11010180). Se leyeron el resumen y el inicio de la introducción en `ProjectESP/Papers/Zhu 2018.pdf`. No se reimplementa su inventario de correlaciones.

**Artículo de surfactante de 2018.** Su tema no se traslada a la serie sin isopropanol. El resto de PDF de la carpeta quedó identificado por la ficha y no se usa para ecuaciones.


## F.3 Enfoque de este laboratorio

Las actividades 2.11 a 2.13 fijaron el diccionario, la normalización y los candidatos. La 2.14 estimó:

- M1, regresión lineal, referencia interpretable.
- M2, polinomio de grado 2 con cresta, referencia no lineal.
- M3, regresión por condición.
- M4, bosque aleatorio. Menor MAE observado: 1,044 psi en mapping y 0,754 psi en surging.
- M5, cociente respecto del agua de entrenamiento, en 5 de 30 curvas de mapping, con 2,078 psi.

`dp2_3_psi` es el incremento de presión de la etapa 3, en psi, con certeza media. No es presión absoluta, altura de columna ni presión de fondo. El protocolo es `2.14-A`. No hay ganador único. El detalle está en el informe de la 2.14.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

def encontrar_raiz_laboratorio() -> Path:
    candidato = Path.cwd().resolve()
    for _ in range(6):
        if (candidato / "data" / "raw" / "mapping.xlsx").is_file():
            return candidato
        if candidato.parent == candidato:
            break
        candidato = candidato.parent
    raise FileNotFoundError("No se encontró el laboratorio.")

RAIZ = encontrar_raiz_laboratorio()
sys.path.insert(0, str(RAIZ / "src"))

from esp_datascience.comparacion_zhu import exigir_comparacion_numerica
from esp_datascience.entrenamiento import HUELLAS_PLIEGUES
from esp_datascience.protocolo import VERSION
from esp_datascience.rutas import huella_sha256

esperadas = {
    "data/raw/mapping.xlsx": "3b933ca84911484f5d912a44b025f5589abaf5ebc2b0426bf80c083869762bda",
    "data/raw/surging.xlsx": "acdcc6c5fc3d8969280c8712c45657fbb863f163a60829584fece5abf75f28cd",
    "data/processed/mapping_long.csv": "ff68e36703d3d3901b8bd964008c49b062afa9d89a817b9d66e7849f9b400648",
    "data/processed/surging_long.csv": "f58e8e2cac74acdf2081479ec8769422f39c4e4665f5b3ff472df4a70b15bd96",
}
for relativo, huella in esperadas.items():
    obtenida = huella_sha256(RAIZ / relativo)
    if obtenida != huella:
        raise RuntimeError(f"Huella distinta en {relativo}")
print("Protocolo", VERSION)
print("Pliegues mapping", HUELLAS_PLIEGUES["mapping"])
print("Pliegues surging", HUELLAS_PLIEGUES["surging"])
print("Las cuatro huellas de archivo coinciden con las publicadas.")


## F.4 Comparabilidad

Qgd no es un caudal de gas en bpd. GVF no es Qgd. `dp2_3_psi` no es la presión de fondo. Un MAE entre métodos solo vale si coinciden el objetivo, las unidades, los datos y el protocolo. Aquí el artículo de 2019 predice fronteras y la actividad 2.14 predice el incremento. La geometría que cierra el modelo mecanicista no está en el Excel. La celda siguiente rechaza la comparación numérica y escribe el juicio cualitativo.


In [ ]:
try:
    exigir_comparacion_numerica(
        objetivo_comun=False,
        unidades_comunes=False,
        protocolo_comun=False,
        ecuacion_verificada=False,
        datos_disponibles=False,
        dominio_conocido=False,
    )
except ValueError as exc:
    print(exc)

juicios = pd.DataFrame([
    ["objetivo", "no comparable", "frontera de patrón frente a dp2_3_psi"],
    ["Qgd y GVF", "no comparable", "Qg/Qmax frente a Qg/(Qg+Ql)"],
    ["velocidad y Psep", "condiciones compartidas", "1800 y 3500 rpm; 50, 100 y 150 psig"],
    ["métrica", "no comparable", "no hay un MAE de Zhu para este incremento"],
    ["banco", "mismo tipo de ensayo", "no se comprobó la identidad punto a punto con las figuras"],
], columns=["aspecto", "juicio", "motivo"])
juicios


## F.5 Tendencias medidas en el CSV, sin reajustar modelos

El gráfico usa solo la hoja de 50 psig y 3500 rpm de mapping. Cada línea es una condición de gas del archivo normalizado. Es una lectura del laboratorio, no una reproducción de la figura 4 de Zhu. Una bajada al aumentar el caudal, dentro de una condición, es una observación de estas filas. No es, por sí sola, la identificación de un régimen.


In [ ]:
import matplotlib.pyplot as plt

mapping = pd.read_csv(RAIZ / "data/processed/mapping_long.csv")
familia = mapping[(mapping["hoja"] == "50psig") & (mapping["velocidad_rpm"] == 3500)].copy()
familia["condicion"] = familia["condicion"].fillna("agua")
fig, ax = plt.subplots(figsize=(8, 4.5))
for condicion, grupo in familia.groupby("condicion"):
    grupo = grupo.sort_values("ql_bpd")
    ax.plot(grupo["ql_bpd"], grupo["dp2_3_psi"], marker="o", label=str(condicion))
ax.set_xlabel("Caudal de líquido (bpd)")
ax.set_ylabel("Incremento de presión de la etapa 3 (psi)")
ax.set_title("Medición en mapping, 50 psig y 3500 rpm")
ax.legend(title="Condición", fontsize=8)
fig.tight_layout()


## F.6 a F.12 Lectura

**F.6 Modelos.** No se codifica la frontera de 2019. Faltan la ecuación aplicada a `dp2_3_psi`, el dominio y la geometría. M1–M5 permanecen como están descritos en la actividad 2.14.

**F.7 Fortalezas.** Zhu aporta el mecanismo y el diseño de los dos ensayos. El laboratorio aporta la comparación de varios estimadores bajo una misma regla de curvas y un intervalo limitado a esas curvas. El menor MAE de M4 no se traduce en mayor capacidad de explicar el régimen.

**F.8 Aporte.** Hay aporte metodológico y predictivo sobre datos publicados por Zhu. No hay evidencia experimental original de este laboratorio.

**F.9 Integración.** Las vías abiertas son un residuo respecto de la curva de agua, una monotonía solo donde el agua la respalde, un indicador de gas alineado con las ecuaciones (2.5) y (2.7), y un mapa de regímenes con métrica distinta de la presión. Ninguna se ajusta aquí.

**F.10 Monitoreo de pozo.** El Excel no trae presión de fondo, nivel dinámico, corriente, potencia ni presión de cabeza. No se declara estimable una variable solo porque exista un balance.

**F.11 Hipótesis.** Quedan en el informe, sección 12, con criterio de respaldo y de rechazo. No son resultados.

**F.12 Cierre.** Los dos enfoques describen el mismo tipo de banco con preguntas distintas. No se sustituyen. La actividad 2.15 sigue abierta hasta una revisión posterior.

La matriz de evidencias distingue lo publicado por Zhu, lo calculado en la 2.14, la inferencia de esta comparación y las hipótesis futuras.


In [ ]:
matriz = pd.read_csv(RAIZ / "docs/matriz_evidencias_2_15.csv")
print(matriz.groupby("tipo_de_evidencia").size().to_string())
print("Filas de la matriz:", len(matriz))
